# IL3.4: Escalabilidad y Sostenibilidad
## Notebook 2: Escalabilidad Horizontal vs Vertical en Agentes de IA

### Objetivo:
Comprender la diferencia de conceptos entre la escalabilidad horizontal y la vertical, y simular la distribución de carga en agentes inteligentes replicados utilizando una lógica de balanceo de carga.

### Estrategias de Crecimiento de Sistemas:
1. **Escalabilidad Vertical (Scale Up):**
   - Consiste en dotar a un único servidor de más recursos físicos (CPU, RAM, almacenamiento).
   - **Ventaja:** Muy sencilla de implementar, no requiere cambios en el software.
   - **Desventaja:** Tiene límites físicos estrictos, costos marginales exponenciales y representa un único punto de fallo (SPOF).
2. **Escalabilidad Horizontal (Scale Out):**
   - Consiste en replicar e iniciar la aplicación (agente) en múltiples servidores que se ejecutan simultáneamente.
   - **Ventaja:** Crecimiento ilimitado, tolerancia a fallos distribuida y uso eficiente del presupuesto de infraestructura.
   - **Desventaja:** Mayor complejidad arquitectónica y necesidad de coordinar llamadas y estados mediante un Balanceador de Carga.


### Inicialización del Entorno
Ejecuta la siguiente celda para configurar el cliente LLM y el agente real de Wikipedia usando LangChain.


In [ ]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = client.pull_prompt("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")


### Réplicas de Agentes Horizontales y Balanceador de Carga
Crearemos múltiples instancias del agente ejecutándose de manera distribuida. Usaremos un balanceador de carga simple que distribuye de forma aleatoria las peticiones del usuario entre las réplicas disponibles.


In [ ]:
import random

class LoadBalancer:
    def select(self, instances):
        # Selección aleatoria de una instancia saludable
        return random.choice(instances) if instances else None

class HorizontalAgent:
    def __init__(self, instance_id, executor):
        self.instance_id = instance_id
        self.executor = executor

    def process(self, request):
        print(f"[Instancia {self.instance_id}] Procesando petición: '{request}'...")
        try:
            if llm is None:
                # Simulación de respuesta
                return f"Respuesta simulada de la instancia {self.instance_id}"
            # Ejecución real del agente Wikipedia
            response = self.executor.invoke({"input": request})
            return f"Instancia {self.instance_id} -> {response.get('output', '')}"
        except Exception as e:
            return f"Error en {self.instance_id}: {e}"

# Crear un pool de 3 réplicas del agente
agent_pool = [
    HorizontalAgent(instance_id=f"agente-replica-{i}", executor=agent_executor)
    for i in range(1, 4)
]

balancer = LoadBalancer()

# Consultas a distribuir
queries = [
    "¿Quién fue Marie Curie?",
    "Explícame el concepto de gravedad.",
    "Busca datos sobre el telescopio espacial Hubble."
]

for query in queries:
    chosen_replica = balancer.select(agent_pool)
    print(f"\n--- Balanceador asignó petición a: {chosen_replica.instance_id} ---")
    print(chosen_replica.process(query))


### Despliegue con Containers (Docker)
En producción, para lograr la escalabilidad horizontal de forma fluida, cada réplica del agente se empaqueta de forma inmutable dentro de un contenedor. A continuación se presenta un ejemplo típico de archivo `Dockerfile` para empaquetar nuestro agente de IA:

```dockerfile
# Imagen base liviana de Python
FROM python:3.9-slim

# Definir variables de entorno
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

WORKDIR /app

# Instalar dependencias necesarias
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiar código fuente
COPY agent.py .

# Comando de ejecución por defecto
CMD ["python", "agent.py"]
```


### Preguntas de Análisis
1. **¿Cuáles son las ventajas de utilizar escalabilidad horizontal (Scale Out) en lugar de escalabilidad vertical (Scale Up) al desplegar agentes de IA para miles de usuarios?**
2. **¿Qué rol cumplen los contenedores (como Docker) y los orquestadores (como Kubernetes) al implementar escalamiento horizontal dinámico?**
3. **¿Cómo manejarías la consistencia de la memoria (el historial de conversación) si un usuario interactúa con diferentes réplicas del agente en cada mensaje de chat?**
